In [2]:
import numpy as np

def gini(y):
    """불순도: 0이면 완벽히 순수, 클수록 섞임"""
    if len(y) == 0:
        return 0.0
    # 각 클래스의 비율 계산
    classes, counts = np.unique(y, return_counts=True)
    # 각 클래스의 갯수(리스트) / 전체 갯수
    # print(classes, counts)
    p = counts / len(y)
    # p: 각 클래스의 비율의 리스트
    # 1 - np.sum(p**2) == 0 : 순수(클래스가 1개) 
    # 0.5 반반
    # print(counts, p)
    # print(np.sum(p ** 2))
    # print(1 - np.sum(p ** 2))
    return 1 - np.sum(p ** 2)

print(gini(np.array([0, 0, 0, 0])))
print(gini(np.array([0, 0, 1, 1])))
print(gini(np.array([0, 0, 0, 1])))
print(gini(np.array([1, 0, 0, 0])))

0.0
0.5
0.375
0.375


In [3]:
def split_gini(X_col, y, threshold):
    """X_col을 threshold 기준으로 좌/우로 나눴을 때의 가중 평균 Gini"""
    left = y[X_col <= threshold] # 기준값 이하 → 왼쪽
    right = y[X_col > threshold] # 기준값 초과 → 오른쪽

    n = len(y)
    # 좌/우 그룹 크기로 가중 평균
    # print(left, len(left) / n, gini(left))
    # print(right, len(right) / n, gini(right))
    weighted = (len(left) / n) * gini(left) + (len(right) / n) * gini(right)
    return weighted

In [4]:
# 피처값과 라벨 (값이 작으면 클래스 0, 크면 클래스 1 - 깔끔히 갈림)
X_col = np.array([1, 2, 3, 8, 9, 10])
y     = np.array([0, 0, 0, 1, 1, 1])

# 3.5에서 자르면? → 왼쪽 [0, 0, 0], 오른쪽 [1, 1, 1] = 완벽 분리
print(split_gini(X_col, y, 3.5)) # 0.0 기대

# 1.5에서 자르면? → 왼쪽 [0], 오른쪽 [0, 0, 1, 1, 1] = 오른쪽 섞임
print(split_gini(X_col, y, 1.5)) # 0보다 큼 기대

0.0
0.4


### print(split_gini(X_col, y, 3.5))
- [0 0 0] 0.5 0.0      ← 왼쪽: 라벨 [0,0,0], 비율 0.5(3/6), gini 0.0(순수)
- [1 1 1] 0.5 0.0      ← 오른쪽: 라벨 [1,1,1], 비율 0.5(3/6), gini 0.0(순수)
- 0.0                  ← weighted = 0.5×0.0 + 0.5×0.0 = 0.0

### print(split_gini(X_col, y, 1.5))
- [0] 0.16666... 0.0           ← 왼쪽: [0] 1개, 비율 1/6, gini 0(순수)
- [0 0 1 1 1] 0.8333... 0.48   ← 오른쪽: 5개 섞임, 비율 5/6, gini 0.48
- 0.4                          ← weighted = (1/6)×0 + (5/6)×0.48 = 0.4

In [5]:
def best_split(X, y):
    """모든 피처 x 모든 기준값을 시도해 Gini가 최소가 되는 분할을 찾는다"""
    best = {"gini": float("inf"), "feature": None, "threshold": None}
    # float("inf"): 무한대. 첫 비교에서 무조건 갱신되도록 시작값을 최대로 둠

    n_features = X.shape[1]
    print(n_features)
    for feat in range(n_features):
        # print("feat", feat)
        thresholds = np.unique(X[:, feat])
        # print("thresholds", thresholds)
        for t in thresholds:
            # print("element", t)
            g = split_gini(X[:, feat], y, t)
            # print(f" 피처{feat}, 기준{t}: gini={g:.4f}")
            if g < best["gini"]:
                best = {"gini": g, "feature": feat, "threshold": t}
                print(best)
    return best

In [6]:
# 피처 2개짜리 장난감 데이터 (6샘플 × 2피처)
# 피처0: 깔끔히 갈림(1,2,3 | 8,9,10), 피처1: 뒤죽박죽(노이즈)
X = np.array([
    [1,  5],
    [2,  9],
    [3,  2],
    [8,  7],
    [9,  1],
    [10, 6],
])
y = np.array([0, 0, 0, 1, 1, 1])

result = best_split(X, y)
print("\n최적 분할:", result)


2
{'gini': np.float64(0.4), 'feature': 0, 'threshold': np.int64(1)}
{'gini': np.float64(0.25), 'feature': 0, 'threshold': np.int64(2)}
{'gini': np.float64(0.0), 'feature': 0, 'threshold': np.int64(3)}

최적 분할: {'gini': np.float64(0.0), 'feature': 0, 'threshold': np.int64(3)}


In [7]:
from sklearn.datasets import load_iris

iris = load_iris()
X = iris.data       # (150, 4) - 붓꽃 150송이 x 속성 3
y = iris.target     # 0=setosa, 1=versicolor, 2=virginica

print("속성 이름:", iris.feature_names)
print("데이터 크기:", X.shape)

# 내 구현으로 첫 분할 찾기 (디버그 print는 많으니 결과만 볼 거면 best_split의 print 주석처리
result = best_split(X, y)
print("\n=== 내 구현이 고른 첫 분할 ===")
print(f"  피처 번호: {result['feature']}")
print(f"  속성 이름: {iris.feature_names[result['feature']]}")
print(f"  기준값: {result['threshold']:.2f}")
print(f"  분할 후 gini: {result['gini']:.4f}")


속성 이름: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
데이터 크기: (150, 4)
4
{'gini': np.float64(0.6621923937360179), 'feature': 0, 'threshold': np.float64(4.3)}
{'gini': np.float64(0.6484018264840183), 'feature': 0, 'threshold': np.float64(4.4)}
{'gini': np.float64(0.6436781609195401), 'feature': 0, 'threshold': np.float64(4.5)}
{'gini': np.float64(0.624113475177305), 'feature': 0, 'threshold': np.float64(4.6)}
{'gini': np.float64(0.6139088729016786), 'feature': 0, 'threshold': np.float64(4.7)}
{'gini': np.float64(0.5870646766169154), 'feature': 0, 'threshold': np.float64(4.8)}
{'gini': np.float64(0.5812026515151516), 'feature': 0, 'threshold': np.float64(4.9)}
{'gini': np.float64(0.5467867231638418), 'feature': 0, 'threshold': np.float64(5.0)}
{'gini': np.float64(0.49824718430670545), 'feature': 0, 'threshold': np.float64(5.1)}
{'gini': np.float64(0.48211640211640217), 'feature': 0, 'threshold': np.float64(5.2)}
{'gini': np.float64(0.47421962095875136)

In [8]:
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(random_state=42)
tree.fit(X, y)

# 트리의 루트 노드(첫 분할, 인덱스 0) 정보
root_feature = tree.tree_.feature[0]      # 첫 분할에 쓴 피처 번호
root_threshold = tree.tree_.threshold[0]  # 첫 분할 기준값

print("=== sklearn이 고른 첫 분할 ===")
print(f"  피처 번호: {root_feature}")
print(f"  속성 이름: {iris.feature_names[root_feature]}")
print(f"  기준값: {root_threshold:.2f}")


=== sklearn이 고른 첫 분할 ===
  피처 번호: 2
  속성 이름: petal length (cm)
  기준값: 2.45


## 결론: 의사결정트리 구현

### 만든 것
| 함수 | 역할 |
|---|---|
| `gini(y)` | 불순도 측정 (0=순수, 0.5=반반) |
| `split_gini(X_col, y, t)` | "이 지점에서 자르면 얼마나 순수해지나" 채점 |
| `best_split(X, y)` | 모든 피처 × 모든 기준값을 완전 탐색해 최저 gini 분할 선택 |

### 검증: 내 구현 ≈ sklearn
iris 첫 분할에서 둘 다 **petal length(피처2)**를 선택 → 일치 ✅
- 기준값만 다름(1.90 vs 2.45): 내 구현은 실제값, sklearn은 인접값의 중간점을 씀.
  자르는 위치는 사실상 동일(그 사이엔 데이터가 없음).

### 배운 것
- 트리는 블랙박스가 아니다 — **gini 채점 + 완전 탐색**이 전부.
- 아무도 알려주지 않아도 **가장 잘 가르는 속성을 스스로 찾는다**
  (노이즈 피처는 gini가 안 내려가 무시됨 → feature_importances_의 뿌리).
- 트리 전체 = 이 best_split을 **재귀로 반복**할 뿐
  (나뉜 각 그룹에 또 best_split → 순수해질 때까지).

### 다음
원리를 이해했으니 재귀·가지치기·예측은 sklearn에 맡긴다.
→ 본격 트리(sklearn DecisionTree)로 titanic 실전:
   스케일링 불필요, 결측/범주형 처리, plot_tree 시각화.
